In [1]:
# ========================================
# SETUP & IMPORTS
# ========================================
# Configure GPU memory and select which GPUs to use
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '7'  # Adjust based on your available GPUs

import json
import torch

# Unsloth - optimized library for efficient LoRA fine-tuning
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, standardize_sharegpt, train_on_responses_only

# Training components
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback
from transformers import TextStreamer

# HuggingFace Hub utilities
from huggingface_hub import delete_repo, create_repo
from datasets import load_dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# ========================================
# VERIFY GPU SETUP
# ========================================
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'Not set')}")
print(f"Number of GPUs PyTorch sees: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

torch.cuda.empty_cache()  # Clear GPU memory from previous runs

CUDA_VISIBLE_DEVICES: 7
Number of GPUs PyTorch sees: 1
  GPU 0: NVIDIA B200


In [3]:
# ========================================
# LOAD BASE MODEL
# ========================================
# We start with a pre-trained model and will add LoRA adapters to it
# LoRA doesn't modify the original model - it adds small trainable layers

max_seq_length = 8192  # Maximum context length (8K tokens)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-32B",  # Base model to fine-tune
    max_seq_length = max_seq_length,
    dtype = None,  # Auto-detect best dtype (bfloat16 if supported, else float16)
    load_in_4bit = False,  # Use full precision for training
    device_map = "auto",  # Automatically distribute model across available GPUs
)

==((====))==  Unsloth 2025.11.3: Fast Qwen3 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA B200. Num GPUs = 1. Max memory: 178.351 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 10.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [4]:
# ========================================
# CHAT TEMPLATE - UNDERSTANDING THE DIFFERENCE
# ========================================
# Chat templates control how conversations are formatted for the model
# This is CRITICAL - the wrong format means poor training results
#
# Let's see what the standard Qwen-3 template does:

tokenizer_test = get_chat_template(
    tokenizer,
    chat_template = "qwen-3",
)

test_conversation = [
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "The answer is 4."}
]

standard_output = tokenizer_test.apply_chat_template(test_conversation, tokenize=False)
print(standard_output)
print("="*50)
print("\nNotice: The standard template adds <think></think> tags")
print("This trains the model to show reasoning steps before answers")

<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
<think>

</think>

The answer is 4.<|im_end|>


Notice: The standard template adds <think></think> tags
This trains the model to show reasoning steps before answers


In [5]:
# ========================================
# CHAT TEMPLATE - CUSTOM (NO THINK TAGS)
# ========================================
# For our use case, we want direct answers without visible reasoning
# WHY? Our training dataset doesn't contain reasoning steps in <think> tags
# If we train with the standard template, the model will expect/generate empty <think> tags
#
# Solution: Custom template that removes <think> tags while keeping ChatML structure
#
# What the template does (it's a Jinja2 template):
# 1. {{ bos_token }} - adds beginning-of-sequence token
# 2. {% for message in messages %} - loops through conversation
# 3. Wraps user messages: <|im_start|>user\n{content}<|im_end|>
# 4. Wraps assistant messages: <|im_start|>assistant\n{content}<|im_end|>
# 5. {% if add_generation_prompt %} - adds <|im_start|>assistant\n when generating

custom_template = """{{ bos_token }}{% for message in messages %}{% if message['role'] == 'user' %}{{ '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}{% elif message['role'] == 'assistant' %}{{ '<|im_start|>assistant\n' + message['content'] + '<|im_end|>\n' }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"""

tokenizer.chat_template = custom_template

# Verify the custom template
custom_output = tokenizer.apply_chat_template(test_conversation, tokenize=False)
print("Custom template output:")
print(custom_output)
print("="*50)
print("\n✓ Clean format - matches our training data structure")

Custom template output:
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
The answer is 4.<|im_end|>


✓ Clean format - matches our training data structure


In [6]:
# ========================================
# CONFIGURE LORA ADAPTERS
# ========================================
# LoRA doesn't modify the base model - it adds small trainable adapter layers
# This means we only train ~100M parameters instead of 32B (much faster!)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,  # LoRA rank - higher = more expressive but slower (try 8, 16, 32, 64)
    
    # Which layers to add LoRA to - attention and MLP projections
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",  # Attention projections
                      "gate_proj", "up_proj", "down_proj",],    # MLP projections
    
    lora_alpha = 32,  # Scaling factor (usually same as rank)
    lora_dropout = 0,  # Dropout for LoRA layers (0 is optimized by Unsloth)
    bias = "none",     # Don't train bias terms (faster, usually works fine)
    use_gradient_checkpointing = "unsloth",  # Saves memory 
    random_state = 3407,   # For reproducibility
    use_rslora = True,     # Rank-Stabilized LoRA - improves training stability
    loftq_config = None,   # LoftQ - quantization-aware initialization (not needed here)
)

Unsloth 2025.11.3 patched 64 layers with 64 QKV layers, 64 O layers and 64 MLP layers.


In [7]:
# ========================================
# LOAD TRAINING DATASET
# ========================================
# We're using ShareGPT format - a standard conversation format for fine-tuning
# ShareGPT structure: list of conversations, each with "from" and "value" fields
# Example: [{"from": "human", "value": "..."}, {"from": "gpt", "value": "..."}]

dataset = load_dataset(
    "json",
    data_files = "Dataset/Processed/Ground Truth/dataset_combined.json",
    split = "train",
)

# Let's look at an example conversation to verify the format
dataset[5]['conversations']

[{'from': 'human',
  'value': 'How do you approach the challenge of maintaining discipline when market conditions soften and competitors start offering unrealistic rates?'},
 {'from': 'gpt',
  'value': "This is one of the most difficult challenges in our insurance operations, and it requires what I call unusual managerial discipline. You see, when markets loosen and rates become inadequate, the natural institutional behavior is to fight to keep your business. Nobody likes to see the other fellow take away their customers. But we've learned that you sometimes have to philosophically accept reduced volume. Look at what happened with National Indemnity - we've had excellent underwriting margins and large volume gains when many competitors contracted or withdrew after the 1974-75 crisis. But these conditions won't last forever. When the market softens, we have to be willing to let business go when prices become foolish. It's like being the designated driver at a party - you might not be ha

In [8]:
# ========================================
# STANDARDIZE DATASET FORMAT
# ========================================
# standardize_sharegpt converts various ShareGPT formats into a consistent structure
# It normalizes role names (human/user → user, gpt/assistant → assistant)
# and field names (from/value → role/content)

dataset = standardize_sharegpt(dataset)

# Check the standardized format - should now have consistent "role" and "content" fields
dataset[5]['conversations']

[{'content': 'How do you approach the challenge of maintaining discipline when market conditions soften and competitors start offering unrealistic rates?',
  'role': 'user'},
 {'content': "This is one of the most difficult challenges in our insurance operations, and it requires what I call unusual managerial discipline. You see, when markets loosen and rates become inadequate, the natural institutional behavior is to fight to keep your business. Nobody likes to see the other fellow take away their customers. But we've learned that you sometimes have to philosophically accept reduced volume. Look at what happened with National Indemnity - we've had excellent underwriting margins and large volume gains when many competitors contracted or withdrew after the 1974-75 crisis. But these conditions won't last forever. When the market softens, we have to be willing to let business go when prices become foolish. It's like being the designated driver at a party - you might not be having as much f

In [9]:
# ========================================
# FORMAT CONVERSATIONS FOR TRAINING
# ========================================
# Apply our chat template to convert conversations into the format the model expects
# Input: [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
# Output: "<|im_start|>user\n...<|im_end|>\n<|im_start|>assistant\n...<|im_end|>"

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, 
            tokenize=False,  # Return strings (tokenize later during training)
            add_generation_prompt=False  # Data already has complete conversations
        )
        for convo in convos
    ]
    return {"text": texts}

# Apply the formatting to all examples
dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = dataset.remove_columns(['conversations'])  # Don't need this anymore

# Verify the formatted output
dataset[5]

{'text': "<|im_start|>user\nHow do you approach the challenge of maintaining discipline when market conditions soften and competitors start offering unrealistic rates?<|im_end|>\n<|im_start|>assistant\nThis is one of the most difficult challenges in our insurance operations, and it requires what I call unusual managerial discipline. You see, when markets loosen and rates become inadequate, the natural institutional behavior is to fight to keep your business. Nobody likes to see the other fellow take away their customers. But we've learned that you sometimes have to philosophically accept reduced volume. Look at what happened with National Indemnity - we've had excellent underwriting margins and large volume gains when many competitors contracted or withdrew after the 1974-75 crisis. But these conditions won't last forever. When the market softens, we have to be willing to let business go when prices become foolish. It's like being the designated driver at a party - you might not be hav

In [10]:
# ========================================
# TRAINING SETUP
# ========================================
# Split data into training and validation sets
# Validation helps us detect overfitting - if eval_loss increases while train_loss decreases, we're overfitting
dataset_split = dataset.train_test_split(test_size=0.15, seed=42)  # 85% train, 15% validation

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_split["train"],
    eval_dataset = dataset_split["test"],
    dataset_text_field = "text",  # Which field contains our formatted conversations
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False,  # Don't pack multiple examples together (cleaner for conversations)
    
    args = TrainingArguments(
        # Batch size and gradient accumulation
        # Effective batch size = per_device_batch_size * gradient_accumulation_steps * num_gpus
        # Here: 2 * 4 * 1 = 8 effective batch size - we are not doing data parallelism
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        
        # Training schedule
        warmup_ratio = 0.05,  # Gradually increase learning rate for first 5% of steps
        num_train_epochs = 2,
        learning_rate = 1e-4,  # LoRA typically uses higher LR than full fine-tuning
        lr_scheduler_type = "cosine",  # Gradually decrease learning rate
        
        # Regularization
        max_grad_norm = 1.0,  # Clip gradients to prevent instability
        weight_decay = 0.01,  # L2 regularization
        
        # Evaluation and checkpointing
        eval_strategy = "steps",
        eval_steps = 100,  # Evaluate every 100 steps
        save_strategy = "steps",
        save_steps = 100,  # Save checkpoint every 100 steps
        save_total_limit = 3,  # Keep only 3 most recent checkpoints
        # load_best_model_at_end = True,  # Uncomment to load best checkpoint at end
        # metric_for_best_model = "eval_loss",
        
        # Precision (use bfloat16 if supported, else float16)
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        
        # Optimizer and logging
        optim = "adamw_torch_fused",  # Faster AdamW implementation
        logging_steps = 10,
        seed = 3407,
        output_dir = "outputs/qwen-buffett-2epoch-split",
        report_to = "none",  # Don't log to wandb/tensorboard
    ),
    
    # callbacks = [EarlyStoppingCallback(early_stopping_patience=3)],  # Uncomment to stop if eval_loss doesn't improve
    callbacks = [],
)

In [11]:
# ========================================
# TRAIN ONLY ON RESPONSES
# ========================================
# By default, the model computes loss on EVERYTHING (user messages + assistant responses)
# But we only want to teach it how to respond - not how to ask questions!
# 
# train_on_responses_only masks out user messages from the loss calculation
# Only the assistant's responses contribute to the loss and gradient updates

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user",  # Start of user message
    response_part = "<|im_end|>",  # End of user message (response starts after this)
)

# Let's see the full tokenized conversation (this includes everything)
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

"<|im_start|>user\nHow does your long-term investment horizon influence your approach to keeping cash reserves?<|im_end|>\n<|im_start|>assistant\nThe key thing to understand is that we have this really pretty unusual flexibility at Berkshire, combined with what I'd emphasize is a very long period ahead of us to find opportunities. We're not thinking quarter to quarter like many others might. We're measuring our cash position in terms of its potential utility over an extended period where we'll have opportunities to deploy funds. This long-term perspective means we can be patient and wait for those moments when we can deploy money at rates that may be quite a bit higher than other people might achieve. It's not just about having cash - it's about having the time horizon and patience to use it optimally. As Charlie points out, we're perfectly willing to pay what amounts to an insurance premium now - keeping substantial cash on hand - to ensure we have a lot of money available when someth

In [12]:
# ========================================
# VISUALIZE WHAT'S ACTUALLY BEING TRAINED
# ========================================
# The "labels" field determines what contributes to loss
# Tokens with label = -100 are masked (not trained on)
# Let's visualize this by replacing masked tokens with spaces

sample = trainer.train_dataset[5]
labels = sample["labels"]

# Replace -100 (masked) with spaces to see what's actually being trained
space = tokenizer(text=" ", add_special_tokens=False).input_ids[0]
filtered = [space if x == -100 else x for x in labels]

# This shows ONLY the assistant's responses - user messages are now spaces
tokenizer.decode(filtered)

"                   \n<|im_start|>assistant\nThe key thing to understand is that we have this really pretty unusual flexibility at Berkshire, combined with what I'd emphasize is a very long period ahead of us to find opportunities. We're not thinking quarter to quarter like many others might. We're measuring our cash position in terms of its potential utility over an extended period where we'll have opportunities to deploy funds. This long-term perspective means we can be patient and wait for those moments when we can deploy money at rates that may be quite a bit higher than other people might achieve. It's not just about having cash - it's about having the time horizon and patience to use it optimally. As Charlie points out, we're perfectly willing to pay what amounts to an insurance premium now - keeping substantial cash on hand - to ensure we have a lot of money available when something really attractive comes up in difficult times. This long-term approach has served us exceptionall

In [13]:
# ========================================
# RUN TRAINING
# ========================================

trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,515 | Num Epochs = 2 | Total steps = 3,130
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 268,435,456 of 33,030,558,720 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,1.585500,1.587971
200,1.553300,1.540591
300,1.463500,1.511759
400,1.442500,1.487625
500,1.405600,1.466748
600,1.413200,1.448952
700,1.434600,1.432470
800,1.419900,1.421893
900,1.367500,1.403458
1000,1.369100,1.391595


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [32]:
# Set model to inference mode
FastLanguageModel.for_inference(model)
# Enable streaming
text_streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

# Test questions

test_questions = [
    # Combine two concepts not usually together
    "How would you think about investing in a business that has strong moats but poor management that refuses to buy back stock even when it's clearly undervalued?",
    
    # Modern scenario Buffett never faced
    "If you were starting your investment career today with $10,000 and access to crypto, AI companies, and traditional businesses, how would you allocate capital?",
    
    # Requires synthesis
    "Can you walk me through a situation where the price of a stock going DOWN would actually make you LESS likely to buy more, despite the business fundamentals staying the same?",
    
    # Counter-intuitive
    "When would high inflation actually be GOOD for certain businesses in your portfolio?"
]

print("="*70)
print("BUFFETTBOT - COMPLEX QUESTION TEST")
print("="*70)

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"QUESTION {i}/{len(test_questions)}:")
    print(f"{'='*70}")
    print(f"{question}\n")
    print(f"{'─'*70}")
    print("ANSWER:")
    print(f"{'─'*70}\n")
    
    # Use the tokenizer's apply_chat_template
    messages = [{"role": "user", "content": question}]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to("cuda")
    
    # Generate with adjusted parameters for longer, more detailed responses
    outputs = model.generate(
        **inputs,
        streamer=text_streamer,
        max_new_tokens=2048,
        temperature=0.85,
        top_p=0.95,
        repetition_penalty=1.05,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    print(f"\n{'='*70}\n")

print("\n" + "="*70)
print("TEST COMPLETE")
print("="*70)

BUFFETT IMPERSONATOR - COMPLEX QUESTION TEST

QUESTION 1/4:
How would you think about investing in a business that has strong moats but poor management that refuses to buy back stock even when it's clearly undervalued?

──────────────────────────────────────────────────────────────────────
ANSWER:
──────────────────────────────────────────────────────────────────────

This is one of the toughest situations you can face as an investor, and it gets right to the heart of what you're really buying when you purchase a stock. You're not just buying a great business; you're buying a partnership with management, and their capital allocation decisions matter enormously.

Let's start with the math. If a business has a million shares outstanding and is worth $100 per share, it's worth $100 million in total. If management buys back 10 percent of the company at $50 per share, they spend $5 million to acquire assets worth $10 million. That's a 2-to-1 value creation, assuming the business value stays

In [ ]:
# ========================================
# SAVE MODEL TO HUGGINGFACE HUB
# ========================================
# We save TWO versions:
# 1. LoRA adapter only (~100MB) - for efficient storage and sharing
# 2. Merged model (~60GB) - ready to use directly for inference

hf_key = "***"
repo = "andreamoccia/BuffettBot"

# Clean up any existing repos (useful for re-running this notebook)
delete_repo(repo_id=repo + "-lora", token=hf_key, missing_ok=True)
delete_repo(repo_id=repo, token=hf_key, missing_ok=True)

# Save LoRA adapter only (~100MB)
# This is lightweight and can be loaded on top of the base model
model.push_to_hub_merged(
    repo + "-lora", 
    tokenizer, 
    save_method="lora",  # Saves ONLY the adapter weights
    commit_message="Initial Commit - LoRA Adapter", 
    commit_description="LoRA adapter after fine-tuning", 
    token=hf_key, 
    private=True,
    create_pr=False
)

# Save merged full model (~60GB)
# This combines base model + LoRA adapter into one model
# Ready to use directly without needing the base model
model.push_to_hub_merged(
    repo, 
    tokenizer, 
    save_method="merged_16bit",  # Merges adapter into base model (16-bit precision)
    commit_message="Initial Commit - Merged Model", 
    commit_description="Full merged model after LoRA fine-tuning", 
    token=hf_key, 
    private=True,
    create_pr=False
)